# In Class Assignment 2006.04.08

## Build a Gaussian Naive Bayes model to predict irrigation need using the data for the Kaggle playground series competition S6E4. You may use your own existing data preprocessing pipeline or adapt from the in-class adult notebook. You should train and test using only the Kaggle train data. 

In [1]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, recall_score, precision_score, balanced_accuracy_score, classification_report
import optuna

In [2]:
df = pd.read_csv("C:/Users/jfigg/OneDrive/Documents/School/GSB 545/Kaggle Competition/Data/train.csv")

In [3]:
df.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [4]:
X = df.drop(["id", "Irrigation_Need"], axis=1)
y_class = df["Irrigation_Need"]
le = LabelEncoder()
y_encoded = le.fit_transform(y_class)

In [5]:
ct_nb = ColumnTransformer([
    ("ohe", OneHotEncoder(), make_column_selector(dtype_include=object))
], remainder="passthrough")
pipeline_np = Pipeline([
    ("preprocessing", ct_nb),
    ("nb", GaussianNB())
])
nb_model = pipeline_np.fit(X, y_encoded)

## Generate predicted probabilities using your model.

In [6]:
nb_probs = nb_model.predict_proba(X)

In [7]:
nb_probs

array([[1.01891367e-20, 9.88735266e-01, 1.12647342e-02],
       [8.09691171e-10, 6.95356107e-01, 3.04643893e-01],
       [2.92410708e-05, 2.31754759e-01, 7.68216000e-01],
       ...,
       [1.21428396e-01, 5.76313287e-03, 8.72808471e-01],
       [2.53912594e-06, 4.73373265e-01, 5.26624195e-01],
       [8.17310797e-11, 9.66742291e-01, 3.32577092e-02]],
      shape=(630000, 3))

## Create baseline predictions using the default rule (choose the class with the highest probability).

In [8]:
nb_pred_class = np.argmax(nb_probs, axis=1)

In [9]:
le.inverse_transform(nb_pred_class)

array(['Low', 'Low', 'Medium', ..., 'Medium', 'Medium', 'Low'],
      shape=(630000,), dtype=object)

In [10]:
print(classification_report(y_encoded, nb_pred_class, target_names=le.classes_))

              precision    recall  f1-score   support

        High       0.53      0.77      0.63     21009
         Low       0.87      0.80      0.83    369917
      Medium       0.70      0.76      0.73    239074

    accuracy                           0.78    630000
   macro avg       0.70      0.78      0.73    630000
weighted avg       0.80      0.78      0.79    630000



## Choose one class to focus on (preferably a rare or important class) and select one evaluation metric (e.g., F1, recall, or balanced accuracy).

In [11]:
y_class = df["Irrigation_Need"]

In [12]:
y_class.value_counts()

Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Choosing the rare "High" value for Irrigation_Need, with the High class `balanced accuracy` as the evaluation metric.

## Apply a threshold to your chosen class by deciding what probability is “high enough” to assign that class. Generate a second set of predictions using this threshold.

In [13]:
def threshold_predictions(probs, threshold, target_class=0):
    other_probs = probs.copy()
    other_probs[:, target_class] = -np.inf
    return np.where(
        probs[:, target_class] >= threshold,
        target_class,
        np.argmax(other_probs, axis=1)
    )

def evaluate_thresholds(y_true, probs, target_class=0, thresholds=None):
    if thresholds is None:
        thresholds=np.linspace(0,1,201)
    rows=[]
    for t in thresholds:
        preds = threshold_predictions(probs, t, target_class=target_class)
        rows.append({
            "threshold":t,
            "precision":precision_score(y_true==target_class,preds==target_class,zero_division=0),
            "recall":recall_score(y_true==target_class,preds==target_class,zero_division=0),
            "f1":f1_score(y_true==target_class,preds==target_class,zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(y_true==target_class,preds==target_class)
        })
    return pd.DataFrame(rows).sort_values("balanced_accuracy", ascending=False)

In [14]:
high_class = le.transform(["High"])[0]
thresholds = evaluate_thresholds(y_encoded, nb_probs, target_class=high_class)

In [15]:
thresholds.head()

,threshold,precision,recall,f1,balanced_accuracy
6,0.030,0.238864,0.895140,0.377100,0.898370
5,0.025,0.230787,0.900233,0.367389,0.898361
7,0.035,0.245677,0.890714,0.385128,0.898184
9,0.045,0.257814,0.883431,0.399144,0.897848
8,0.040,0.251808,0.886525,0.392213,0.897827


In [16]:
best_threshold = thresholds.iloc[0]["threshold"]
print(f"Best Threshold: {best_threshold}")

Best Threshold: 0.03


## Using a markdown cell, discuss which class you selected, what threshold you chose, how the metric changed, what tradeoff you observed, and how Naive Bayes compares to your existing model(s).  You should use a classification report to assist with this discussion.

I selected the rarest class, when `Irrigation_Need = High`, and tuned the probability threshold for that class. The best threshold by High-class balanced accuracy was `.03`. This makes the model much more willing to predict `High`, which increases High recall but lowers High precision and overall accuracy. That tradeoff is expected when tuning for a rare class instead of optimizing overall accuracy.

In [17]:
y_preds = threshold_predictions(nb_probs, best_threshold, target_class=high_class)
print(classification_report(y_encoded, y_preds, target_names=le.classes_))

              precision    recall  f1-score   support

        High       0.24      0.90      0.38     21009
         Low       0.87      0.80      0.83    369917
      Medium       0.65      0.58      0.61    239074

    accuracy                           0.72    630000
   macro avg       0.59      0.76      0.61    630000
weighted avg       0.77      0.72      0.73    630000



The classification report with the new threshold shows the tradeoff from tuning for the rare `High` class. Compared with the default argmax predictions, High recall increases from about `.77` to about `.90`, but High precision drops from about `.53` to about `.24`. Overall accuracy falls from about `.78` to about `.72`, and weighted F1 falls from about `.79` to about `.73`, so the threshold is only better if finding more `High` cases is more important than avoiding false alarms.